In [ ]:
import pandas as pd
import numpy as np
from influxdb_client_3 import InfluxDBClient3, Point

In [ ]:
client = InfluxDBClient3(token=f"apiv3_hoBjjFB7ZYYtYy33a_Ui5VTjsWG848623y8Mlf_a_2gVU86YB8gwVjJVYRJXaTUWnS4desa23-y7DcXa4qxmIg", 
                         host=f"http://localhost:8181", database=f"jkp_factors")

query = f"select ret_exc, time from characteristics WHERE gvkey='184996'"

reader = client.query(query=query, language="sql")
df = reader.to_pandas()
stock = df.dropna().sort_values("time")
stock

In [ ]:
def factor_returns(name):
    query = f"select ret, time from factor_returns where name='{name}'"
    return client.query(query=query, language="sql").to_pandas().rename(columns={"ret":f"ret_{name}"})

In [ ]:
import statsmodels.api as sm
from functools import reduce

factors = ["value", "momentum", "size"]
factor_data = [factor_returns(factor) for factor in factors]

merged_x = reduce(lambda left, right: pd.merge(left, right, on="time"), factor_data)
merged = pd.merge(merged_x, stock, on="time")

X = sm.add_constant(merged.drop(columns=["ret_exc", "time"]))
y = merged["ret_exc"]

model = sm.OLS(y, X).fit()
model.summary()

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(merged["ret_exc"], merged["ret_value"])